In [0]:
K.<w> = QuadraticField(-2, embedding=CC(I)*sqrt(CC(2)))
OK = K.ring_of_integers()
U = K.unit_group()

print(OK)

In [0]:
def closest_integer(gamma):
    K = gamma.parent()
    OK = K.ring_of_integers()
    
    # 1. Get a basis for the ring of integers
    # This basis is {1, omega}
    basis = OK.basis()
    
    # 2. Embed the lattice and target into R^2
    def embed(element):
        c = CC(element) 
        return vector(RR, [c.real(), c.imag()])

    
    # B is the basis matrix
    B = matrix(RR, [embed(b) for b in basis]).transpose()
    target = embed(gamma)
    
    # Get exact coordinates in the lattice basis
    coords = B.solve_right(target)
    
    # 3. Test all 4 combinations of floor and ceiling for the 2 coordinates
    best_dist = infinity
    best_integer = None
    
    import itertools
    # Defines Nearest Neighbours with integer coefficients (note simple rounding would not work for oblique lattices)
    options = [[floor(c)-1, floor(c), floor(c)+1, ceil(c), ceil(c)+1] for c in coords]
    for p, q in itertools.product(*options):
        candidate = OK([p, q])
        # Calculate Euclidean distance squared in the embedding space
        diff = embed(candidate) - target
        dist_sq = diff.dot_product(diff)
        
        if dist_sq < best_dist:
            best_dist = dist_sq
            best_integer = candidate
            
    return best_integer

# test value below for example
gamma = K(-w-1).quo_rem(K(-w))[0]
print(f"Target: {gamma}")
print(f"Closest Integer: {closest_integer(K(gamma))}")

In [0]:
units = [1, -1]

print(f"units of norm 1: {K.elements_of_norm(1)}")

N0 = sorted({u*n for u in units for n in K.elements_of_norm(0)});
N1 = sorted({u*n for u in units for n in K.elements_of_norm(1)});
N2 = sorted({u*n for u in units for n in K.elements_of_norm(2)});
N3 = sorted({u*n for u in units for n in K.elements_of_norm(3)});

S = N0 + N1 + N2 + N3;
T = N1 + N2 + N3;

In [0]:
mats = []
matsupdate = []
for a in S:
    for b in S:
        for c in T:
           for d in S:
               M = matrix(K, [[a, b], [c, d]])
               if M.det() == 1:
                    theta = closest_integer(K(K(-d).quo_rem(K(c))[0])) # closest integer of quotient
                    alpha = K(theta)*K(a) + K(b) # update entry a
                    gamma = K(theta)*K(c) + K(d) # update entry c
                    N = matrix(K, [[alpha, -a], [gamma, -c]])
                    mats.append(M)                 
                    matsupdate.append(N)

# Number and sample
# len(mats), mats
 



# Number and sample
# len(mats), mats

In [0]:
def max_abs_squared_norm(M):
    """
    Return max of the norms of the elements in the 2x2 matrix M. Works because we are in a quadratic field
    """
    return max((x*x.galois_conjugate() for x in M.list()))

normmats = []
for m in mats:
    normmats.append(max_abs_squared_norm(m))
    
normmatsupdate = []
for m in matsupdate:
    normmatsupdate.append(max_abs_squared_norm(m))
    
# len(normmatsupdate), normmatsupdate
    
# len(normmats), normmats

In [0]:
result = list(zip(mats, matsupdate, normmats, normmatsupdate))
 
for i, (M, N, x, y) in enumerate(result, 1):
    print(f"#{i:3d} matrix {list(map(list, M.rows()))} update {list(map(list, N.rows()))} | norms=({x}, {y})")

In [0]:
for i, (M, N, x, y) in enumerate(result, 1):
    if x < y:
        print(f"#{i:3d} matrix {list(map(list, M.rows()))} update {list(map(list, N.rows()))} | norms=({x}, {y})")

The above gives the cases where the closest integer approximation consists of two equidistant points and the sage implementation choices the \`wrong' point for the division algorithm. In all the above cases we have \(w\+1\) // w \(up to sign\).  These should give a quotient 1, but instead give 1 \+ w \(again up to signs\)
